# Practice 27 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: `category` dtype

### What is `category` dtype?

By default, string columns in pandas are stored as `object` dtype — each value is a full Python string object in memory. If a column has many repeated values (e.g. `'male'`/`'female'`, `'Ideal'`/`'Premium'`/...), this wastes a lot of memory.

`category` dtype stores each unique value **once** and represents each row as a small integer code pointing to that value — much more efficient.

**Converting:**
```python
df['col'] = df['col'].astype('category')
```

**Checking memory:**
```python
df.memory_usage(deep=True)   # deep=True counts actual string sizes
```

**The `.cat` accessor** gives you category-specific tools:
```python
df['col'].cat.categories   # the unique values
df['col'].cat.codes        # integer code for each row
```

**Ordered categories** let you use `>`, `<` comparisons:
```python
cut_order = pd.CategoricalDtype(
    categories=['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'],
    ordered=True
)
df['cut'] = df['cut'].astype(cut_order)
df[df['cut'] > 'Good']   # works because it's ordered
```

### Task 1: diamonds — category dtype

Load `sns.load_dataset('diamonds')`.

1. Check the memory usage of the full DataFrame before any changes. Which column uses the most memory?
2. Convert all `object` columns to `category` dtype. Check memory usage again — how much did the total shrink?
3. For the `cut` column, assign an **ordered** `CategoricalDtype` with the correct quality order: `Fair < Good < Very Good < Premium < Ideal`. How many diamonds have a cut **strictly better than** `'Good'`?
4. Print the `.cat.codes` for the first 5 rows of `cut`. What do the codes represent?
5. Use a list comprehension to collect the names of all columns that are now `category` dtype. Print the list.

In [10]:
# Your code here
diamonds = sns.load_dataset('diamonds')



In [127]:
diamonds.memory_usage(deep=True)
### numeric columns use the most memory

diamonds['cut'] = diamonds['cut'].astype('category')
diamonds['color'] = diamonds['color'].astype('category')
diamonds['clarity'] = diamonds['clarity'].astype('category')

diamonds.memory_usage(deep=True)

### diamonds originally is already in categorical 

cut_order = pd.CategoricalDtype(
    categories=['Fair','Good','Very Good','Premium','Ideal'],
    ordered=True
)
diamonds['cut'] = diamonds['cut'].astype(cut_order)
bg = diamonds[diamonds['cut']>'Good']
bg.shape[0]

diamonds['cut'].cat.codes[0:5]

diamonds.select_dtypes(include='category').columns
### step 5, don't know how to do list comprehension

cn = [col for col in diamonds.columns if diamonds[col].dtype.name == 'category']


In [18]:
diamonds.head(5)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


---
## Level 2 — Transformations

### Task 2: student grades — reshape and compare

A wide DataFrame of student grades across terms is provided.

- Stack to long format. Name the value column `'grade'`.
- Add `z_score` per term (standardize within each term using `transform`).
- Unstack so terms are columns. Which student is most consistently above average (highest minimum z-score across all terms)?
- Use `.xs()` to extract the full z-score profile for `'T2'`.
- Build a dict mapping each student to their best term (highest grade). Use a loop or dict comprehension — no `.idxmax()`.

In [42]:
# Starter data — don't change this
np.random.seed(42)
students = ['Amir', 'Beth', 'Carlos', 'Dana', 'Eli', 'Fiona']
terms = ['T1', 'T2', 'T3', 'T4']
grades_wide = pd.DataFrame(
    np.random.randint(55, 100, size=(len(students), len(terms))),
    index=students,
    columns=terms
)
grades_wide.index.name = 'student'

# Your code here
grades_long = grades_wide.stack().to_frame('grade')
#grades_long.index.names = ['student','term']
m = grades_long.groupby(level=1)['grade'].transform('mean')
s = grades_long.groupby(level=1)['grade'].transform('std')
grades_long['z_score'] = (grades_long['grade']-m)/s


us = grades_long['z_score'].unstack(level=1)
us.min(axis=1).idxmax()






grades_long.xs('T2', level=1)

s_bt = {}
for sd in grades_long.index.get_level_values('student').unique():
    s = grades_long.xs(sd,level='student').copy()
    s_bt[sd] = s.index[np.argmax(s['grade'])]

s_bt


{'Amir': 'T4',
 'Beth': 'T3',
 'Carlos': 'T4',
 'Dana': 'T2',
 'Eli': 'T4',
 'Fiona': 'T2'}

---
## Level 3 — Aggregation

### Task 3: tips — pipeline with category

Load `sns.load_dataset('tips')`.

Build a `.pipe()` chain with exactly 3 functions:

- One must convert at least two appropriate columns to `category` dtype
- One must use `transform` to add a group-level metric
- One must classify using `.apply()`

After the pipeline:
- Check the memory usage before and after — how much did converting to `category` save?
- Set a `(day, time)` MultiIndex. Use `.xs()` to retrieve all Dinner rows.

In [53]:
# Your code here

tips = sns.load_dataset('tips')

def convert_c(df):
    df['smoker'] = df['smoker'].astype('category')
    df['sex'] = df['sex'].astype('category')
    return df
def sex_avg_tip(df):
    df['sex_avg_tip'] = df.groupby('sex', observed = True)['tip'].transform('mean')
    return df
def meal_class(df):
    df['meal_class'] = df['total_bill'].apply(lambda x: 'big' if x>50
                                              else 'small')
    return df

res = (
    tips.copy()
    .pipe(convert_c)
    .pipe(sex_avg_tip)
    .pipe(meal_class)
)

res.memory_usage(deep=True)


res = res.set_index(['day','time'])

res.xs('Dinner', level='time')

,total_bill,tip,sex,smoker,size,sex_avg_tip,meal_class
day,,,,,,,
Sun,16.99,1.01,Female,No,2,2.833448,small
Sun,10.34,1.66,Male,No,3,3.089618,small
Sun,21.01,3.50,Male,No,3,3.089618,small
Sun,23.68,3.31,Male,No,2,3.089618,small
Sun,24.59,3.61,Female,No,4,2.833448,small
...,...,...,...,...,...,...,...
Sat,29.03,5.92,Male,No,3,3.089618,small
Sat,27.18,2.00,Female,Yes,2,2.833448,small
Sat,22.67,2.00,Male,Yes,2,3.089618,small


---
## Level 4 — Real-world

### Task 4: titanic — memory audit + open analysis

Load `sns.load_dataset('titanic')`.

No scaffolding. Five questions:

1. Which columns have more than 10% null values? Collect them into a list using a list comprehension.
2. Convert every column with fewer than 10 unique values to `category` dtype. How much memory did you save in total?
3. Using the ordered category concept — define an ordered dtype for `pclass` (`3 < 2 < 1`). How does mean survival rate change as class increases? Use NumPy, not groupby.
4. Build a `.pipe()` chain of your choosing on the cleaned DataFrame. At least one function must use the `category` dtype you created (e.g. filter using ordered comparison, or group by a category column).
5. Create a reshaped view showing survival rate by `(pclass, sex)`. Use `.xs()` to extract one slice.

In [ ]:
# Your code here

titanic = sns.load_dataset('titanic')
titanic_na = titanic.isnull()
titanic_na_count = titanic_na.sum()
titanic_na_count[titanic_na_count>titanic.shape[0]*0.1]
### don't know what list comprehension is 
[col for col in titanic.columns if titanic[col].isnull().mean() > 0.1]


c_u = {}
for col in titanic.columns:
    c_u[col] = len(titanic[col].unique())

## select columns under 10 unique values 
change_col = [k for k, v in c_u.items() if v <10]
titanic_new = titanic.copy()
titanic_new[change_col] = titanic_new[change_col].astype('category')
titanic_new.dtypes
titanic.memory_usage(deep=True).sum() - titanic_new.memory_usage(deep=True).sum()


pclass = pd.CategoricalDtype(
    categories=[3,2,1],
    ordered=True
)
titanic_new['pclass'] = titanic_new['pclass'].astype(pclass)
surv1 = titanic_new['survived'][titanic_new['pclass']==1]
surv2 = titanic_new['survived'][titanic_new['pclass']==2]
surv3 = titanic_new['survived'][titanic_new['pclass']==3]
print(np.mean(surv1.astype(float)), np.mean(surv2.astype(float)), np.mean(surv3.astype(float)))

0.6296296296296297 0.47282608695652173 0.24236252545824846


In [124]:
(titanic_new['pclass']>3).sum()


np.int64(400)

In [125]:
def num_notWorst(df):
    return (df['pclass']>3).sum()

def flag_old(df):
    df['old'] = df['age']>60
    return df 

res = (
    titanic_new.copy()
    .pipe(flag_old)
    .pipe(num_notWorst)
)
res

np.int64(400)

In [113]:
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [110]:
titanic_new['survived'] = titanic_new['survived'].astype(float)
p = titanic_new.pivot_table(
    index = 'pclass',
    columns = 'sex',
    values = 'survived'
).stack()
p.xs(3,level = 'pclass').xs('female')

/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_28335/3361176050.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  p = titanic_new.pivot_table(


np.float64(0.5)

In [111]:
p


pclass  sex   
3       female    0.500000
        male      0.135447
2       female    0.921053
        male      0.157407
1       female    0.968085
        male      0.368852
dtype: float64